
# Week 3 — Gemma 3 + Shadow-FT on Colab（免費 T4）

**這份 notebook 對應 `week3_執行手冊.md`。手冊講「為什麼」，這裡是「怎麼跑」。**

---

## 開跑前必讀

| 項目 | 值 | 為什麼 |
|---|---|---|
| Runtime | **T4 GPU**（選單：執行階段 → 變更執行階段類型 → T4 GPU） | 免費版唯一選項 |
| 單次上限 | 12 小時 | 中途斷線會全部消失 → **所有產出都寫進 Drive** |
| 每週配額 | 約 15–30 GPU 小時（Google 不公布確切數字，會浮動） | 本 notebook 全跑約 8–10 小時，留有餘裕 |
| bf16 | **沒有**。T4 是 Turing，只有 fp16 | Gemma 3 在 fp16 下 layernorm 後 activation 可達 ~800,000 > fp16 上限 65,504 → `inf` → `NaN`。這就是必須用 Unsloth 的唯一理由 |

**斷點續跑**：每個實驗跑完會在 Drive 寫一個 `done.json`。重跑整份 notebook 時已完成的會自動跳過。所以斷線之後：重連 → 從頭 Run All → 它會接著上次的進度。

---

## 執行順序

```
§1 環境      →  §2 資料      →  §3 記憶體預測（先算再量，沿用 Week 1/2 的做法）
     ↓
§4 框架對照（HF vs Unsloth，各 30 步）        ← 答主管 Q1
     ↓
§5 Stage A/B/C：LoRA 參數掃描                 ← 答主管 Q2
     ↓
§6 Shadow-FT（在 -pt 上訓練，把 adapter 搬到 -it）  ← 答主管 Q4
     ↓
§7 TMMLU+ 評測（種子固定、macro + micro 並列）
     ↓
§8 彙整成表與圖
```


---
# §1 環境

## 1.1 確認拿到什麼卡

In [ ]:
#@title 1.1 GPU / 環境檢查
import subprocess, sys, os, platform, json

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

try:
    import torch
    p = torch.cuda.get_device_properties(0)
    cc = f"{p.major}.{p.minor}"
    print(f"GPU            : {p.name}")
    print(f"VRAM           : {p.total_memory/2**30:.2f} GiB")
    print(f"Compute cap.   : {cc}")
    print(f"支援 bf16      : {torch.cuda.is_bf16_supported()}")
    print(f"torch          : {torch.__version__}")
    if not torch.cuda.is_bf16_supported():
        print()
        print("→ 沒有 bf16（T4 / Turing）。Gemma 3 在純 fp16 下會溢位成 NaN，")
        print("  必須用 Unsloth 的混合精度修補。§4 會實測給你看。")
except Exception as e:
    print("!! 沒有 GPU：執行階段 → 變更執行階段類型 → T4 GPU")
    raise


## 1.2 安裝

Unsloth 的相依樹在 Colab 上很敏感，照官方建議的兩段式裝法（先裝完整版拉相依，再用 `--no-deps` 覆蓋成最新）。

In [ ]:
#@title 1.2 安裝套件（約 3–5 分鐘，只需跑一次）
%%capture
import os
IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    !pip install -q unsloth
    !pip install -q --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo
    # 評測與資料處理
    !pip install -q "datasets>=3.2.0" "pandas>=2.3.0" pyarrow pyyaml datasketch matplotlib

In [ ]:
#@title 1.2b 驗收：每一項都要能 import
import importlib, sys
need = ["torch", "transformers", "peft", "trl", "datasets", "bitsandbytes", "unsloth", "pandas", "pyarrow"]
bad = []
for m in need:
    try:
        mod = importlib.import_module(m)
        print(f"  ok   {m:<14} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"  FAIL {m:<14} {type(e).__name__}: {e}")
        bad.append(m)
assert not bad, f"缺套件：{bad} —— 重跑 1.2，若仍失敗則「執行階段 → 重新啟動工作階段」後再跑一次"
print("\n全部就緒")


## 1.3 掛 Drive、決定路徑

**所有產出都寫進 Drive。**Colab 的本機磁碟在斷線後會清空，8 小時的實驗會歸零。

In [ ]:
#@title 1.3 掛 Drive 並建立目錄
import os, json, time
from pathlib import Path

USE_DRIVE = True  #@param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/ultrascale-lab-week3')
else:
    ROOT = Path('/content/ultrascale-lab-week3')

for sub in ['data', 'out', 'results', 'reports', 'datasets', 'logs']:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

# 大檔（模型權重、adapter）放本機碟，只把小的結果檔同步回 Drive。
# 原因：Drive 的 I/O 很慢，訓練時每步寫 checkpoint 會拖垮速度。
SCRATCH = Path('/content/scratch')
for sub in ['out', 'hf_cache']:
    (SCRATCH / sub).mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(SCRATCH / 'hf_cache')

print("結果（要保留）:", ROOT)
print("暫存（可丟棄）:", SCRATCH)


## 1.4 HuggingFace 登入

`google/gemma-3-*` 需要接受授權條款。**兩個都要接受**（`-pt` 和 `-it` 是分開的 repo）：

- https://huggingface.co/google/gemma-3-4b-it
- https://huggingface.co/google/gemma-3-4b-pt

然後在 Colab 左側的「🔑 密鑰」加一個叫 `HF_TOKEN` 的密鑰（token 從 https://huggingface.co/settings/tokens 產生，read 權限即可），並把下一格的 `USE_OFFICIAL_REPO` 勾成 `True`。

**預設是 `False`，用 `unsloth/gemma-3-4b-*` 鏡像 —— 權重與官方相同、不需要接受授權條款。**沒有 token 的話直接跑就好。

（`USE_OFFICIAL_REPO=True` 但找不到 token 會直接報錯，不會安靜地退回鏡像 —— 否則你會以為在跑官方權重，其實不是。）

In [ ]:
#@title 1.4 登入並決定用哪個 repo
from huggingface_hub import login
import os

tok = None
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception:
    tok = os.environ.get('HF_TOKEN')

USE_OFFICIAL_REPO = False  #@param {type:"boolean"}
# False = unsloth 鏡像（權重相同、不需授權，預設）
# True  = google 官方 repo（需要 HF_TOKEN 且兩個 repo 的條款都已接受）

if tok:
    login(token=tok)
    print("已登入 HF。")
elif USE_OFFICIAL_REPO:
    raise RuntimeError("USE_OFFICIAL_REPO=True 但找不到 HF_TOKEN。"
                       "請在左側「🔑 密鑰」新增 HF_TOKEN，或把它改回 False。")

if USE_OFFICIAL_REPO:
    MODEL_IT, MODEL_PT = "google/gemma-3-4b-it", "google/gemma-3-4b-pt"
else:
    MODEL_IT, MODEL_PT = "unsloth/gemma-3-4b-it", "unsloth/gemma-3-4b-pt"
    print("使用 unsloth 鏡像（不需授權，權重與官方相同）。")

# 4-bit 量化版（下載快 4 倍，訓練時本來就會量化）
MODEL_IT_4BIT = MODEL_IT + "-bnb-4bit"
MODEL_PT_4BIT = MODEL_PT + "-bnb-4bit"

print(f"INSTRUCT : {MODEL_IT}")
print(f"BASE     : {MODEL_PT}   ← Shadow-FT 在這個上面訓練")


---
# §2 資料

Week 2 用的是 `twinkle-ai/tw-reasoning-instruct-50k`，8,000 筆抽樣（train 7,600 / val 400），seed 42。這裡完全沿用**同一份抽樣**，只換渲染模板。

## 2.1 一個必須做的決定：`think` 欄位怎麼渲染

Gemma 4 有 thinking channel，Week 2 把 CoT 放進 `<|channel>thought`。**Gemma 3 沒有這個東西。**三個選項：

| 選項 | 做法 | 後果 |
|---|---|---|
| **`inline`（預設）** | 把 think 用 `<think>…</think>` 包起來，放在回答前面，都在 model turn 裡 | 最接近 Week 2；但也最容易複製 Week 2 的格式崩潰（模型學會「先長篇思考再散文回答」） |
| `drop` | 完全丟掉 think，只留最終回答 | 訓練訊號變短變乾淨，格式風險最低；但丟掉了資料集最有價值的部分 |
| `inline_mixed` | `inline` + 混入 5% 的 `\box{X}` 格式樣本（取自 TMMLU+ **訓練科目**，與評測的三科不重疊） | Week 2 總結第 4.4 節的修法 B。**推薦**：能同時測「格式崩潰」和「混入格式樣本能不能救」 |

**這一格會全部產生**，之後每個實驗指定要用哪一份。

In [ ]:
#@title 2.1 下載並渲染訓練資料
import json, random, re
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer

N_SAMPLE   = 8000   #@param {type:"integer"}
VAL_RATIO  = 0.05   #@param {type:"number"}
SEED       = 42     #@param {type:"integer"}
MAX_CHARS  = 8000   #@param {type:"integer"}

DATA = ROOT / 'data'
tok_it = AutoTokenizer.from_pretrained(MODEL_IT)   # ← 一律用 INSTRUCT 的 tokenizer/模板

# ---- Gemma 3 的 chat template 支不支援 system role？先測，不要假設 ----
def supports_system(tk):
    try:
        tk.apply_chat_template(
            [{"role": "system", "content": "x"}, {"role": "user", "content": "y"}],
            tokenize=False, add_generation_prompt=True)
        return True
    except Exception:
        return False

HAS_SYSTEM = supports_system(tok_it)
print(f"Gemma 3 chat template 支援 system role: {HAS_SYSTEM}")

def render(user, assistant, system=None):
    # 回傳訓練用的純文字。注意結尾不含 generation prompt。
    msgs = []
    if system:
        if HAS_SYSTEM:
            msgs.append({"role": "system", "content": system})
        else:
            user = system + "\n\n" + user      # 退回：折進第一個 user turn
    msgs.append({"role": "user", "content": user})
    msgs.append({"role": "assistant", "content": assistant})
    text = tok_it.apply_chat_template(msgs, tokenize=False)
    # 【坑】apply_chat_template 會在最前面加 <bos>，而 SFTTrainer 之後 tokenize 時
    #      預設 add_special_tokens=True 又會再加一個 → 雙 BOS，訓練與推論的前綴不一致。
    #      這裡先剝掉，讓 tokenizer 統一負責。
    if tok_it.bos_token and text.startswith(tok_it.bos_token):
        text = text[len(tok_it.bos_token):]
    return text

if not (DATA / 'train_inline.jsonl').exists():
    raw = load_dataset("twinkle-ai/tw-reasoning-instruct-50k", split="train")
    print(f"step1 載入      : {len(raw):,}")

    def ok(r):
        if not (r.get('input') and r.get('output')): return False
        n = len(r.get('input','')) + len(r.get('think') or '') + len(r.get('output',''))
        return 20 <= n <= MAX_CHARS
    raw = raw.filter(ok)
    print(f"step2 清理      : {len(raw):,}")

    seen, keep = set(), []
    for i, r in enumerate(raw):
        k = r['input'].strip()
        if k in seen: continue
        seen.add(k); keep.append(i)
    raw = raw.select(keep)
    print(f"step3 精確去重  : {len(raw):,}")

    raw = raw.shuffle(seed=SEED).select(range(min(N_SAMPLE, len(raw))))
    n_val = int(len(raw) * VAL_RATIO)
    splits = {'valid': raw.select(range(n_val)), 'train': raw.select(range(n_val, len(raw)))}
    print(f"step4 抽樣切分  : train {len(splits['train']):,} / valid {len(splits['valid']):,}")

    for mode in ('inline', 'drop'):
        for name, ds in splits.items():
            out = DATA / f'{name}_{mode}.jsonl'
            with out.open('w') as f:
                for r in ds:
                    think = (r.get('think') or '').strip()
                    ans = r['output'].strip()
                    body = f"<think>\n{think}\n</think>\n\n{ans}" if (mode == 'inline' and think) else ans
                    f.write(json.dumps({"text": render(r['input'].strip(), body)}, ensure_ascii=False) + "\n")
            print(f"  寫出 {out.name:<22} {sum(1 for _ in out.open()):,} 筆")
else:
    print("資料已存在，跳過。要重做請刪掉 data/*.jsonl")

print()
print("=== 一筆完整樣本（前 700 字）===")
print(json.loads((DATA / 'train_inline.jsonl').open().readline())['text'][:700])


## 2.2 混入 `\box{}` 格式樣本（`inline_mixed`）

Week 2 總結 4.4 節的修法 B。**關鍵是科目不能重疊**：格式樣本只從 TMMLU+ 的**其他科目**抽，評測用的那三科（台灣地理、台語、三民主義）一題都不能碰，否則就是資料洩漏。

In [ ]:
#@title 2.2 產生 inline_mixed（混入 5% 格式樣本）
import pandas as pd, random, json
from datasets import load_dataset

MIX_RATIO = 0.05  #@param {type:"number"}
EVAL_SUBJECTS = ["geography_of_taiwan", "taiwanese_hokkien", "three_principles_of_people"]

SYS_BOX = (
    "使用者將提供一個題目，並附上選項 A、B、C、D。\n"
    "請仔細閱讀題目要求，根據題意選出最符合的選項，並將選項以以下格式輸出：\n"
    "\\box{選項}\n"
    "請確保僅將選項包含在 { } 中，否則將不計算為有效答案。\n"
    "務必精確遵循輸出格式，避免任何多餘內容或錯誤格式。\n"
    "例如：答案是 A，就輸出 \\box{A}。\n"
)

TMMLU_DIR = ROOT / 'datasets' / 'tmmluplus'
TMMLU_DIR.mkdir(parents=True, exist_ok=True)

# ---- 下載 TMMLU+ ----
if not any(TMMLU_DIR.glob("*.parquet")):
    from huggingface_hub import snapshot_download
    p = snapshot_download(repo_id="ikala/tmmluplus", repo_type="dataset",
                          local_dir=str(TMMLU_DIR / '_raw'))
    import shutil
    for f in Path(p).rglob("*test*.csv"):
        pd.read_csv(f).to_parquet(TMMLU_DIR / (f.stem.replace('_test','') + '.parquet'))
    for f in Path(p).rglob("*.parquet"):
        if f.parent != TMMLU_DIR:
            shutil.copy(f, TMMLU_DIR / f.name)
subjects = sorted(f.stem for f in TMMLU_DIR.glob("*.parquet"))
print(f"TMMLU+ 科目數：{len(subjects)}")
assert len(subjects) >= 10, "TMMLU+ 下載失敗，檢查上一步"

# ---- 抽格式樣本（排除評測科目）----
if not (DATA / 'train_inline_mixed.jsonl').exists():
    pool = [s for s in subjects if s not in EVAL_SUBJECTS]
    rng = random.Random(SEED)
    n_train = sum(1 for _ in (DATA / 'train_inline.jsonl').open())
    n_mix = int(n_train * MIX_RATIO / (1 - MIX_RATIO))

    rows = []
    for s in pool:
        df = pd.read_parquet(TMMLU_DIR / f"{s}.parquet")
        for _, r in df.iterrows():
            if all(k in r and pd.notna(r[k]) for k in ("question","A","B","C","D","answer")):
                rows.append(r)
    rng.shuffle(rows)
    rows = rows[:n_mix]

    with (DATA / 'train_inline_mixed.jsonl').open('w') as f:
        for line in (DATA / 'train_inline.jsonl').open():
            f.write(line)
        for r in rows:
            q = r['question'] + "\n" + "\n".join(f"{k}: {r[k]}" for k in "ABCD")
            f.write(json.dumps({"text": render(q, f"\\box{{{str(r['answer']).strip().upper()}}}",
                                               system=SYS_BOX)}, ensure_ascii=False) + "\n")
    import shutil; shutil.copy(DATA/'valid_inline.jsonl', DATA/'valid_inline_mixed.jsonl')
    print(f"混入 {len(rows):,} 筆格式樣本（{100*len(rows)/(n_train+len(rows)):.1f}%），"
          f"來自 {len(pool)} 個非評測科目")
else:
    print("inline_mixed 已存在，跳過。")

# ---- 防呆：格式樣本絕不能來自評測科目 ----
for s in EVAL_SUBJECTS:
    df = pd.read_parquet(TMMLU_DIR / f"{s}.parquet")
    qs = set(df['question'].astype(str))
    hit = sum(1 for l in (DATA/'train_inline_mixed.jsonl').open()
              if any(q[:40] in l for q in list(qs)[:200]))
    assert hit == 0, f"資料洩漏！{s} 的題目出現在訓練集裡"
print("防呆通過：評測三科的題目沒有出現在訓練資料中")


---
# §3 記憶體預測（先算再量）

沿用 Week 1／Week 2 的做法：**先用公式算出預測值，再去量，然後解釋差在哪。**Week 2 最大的收穫之一就是「公式方向對，但直接代入會系統性高估」。

Gemma 3 4B 這裡有一個和 Week 2 一模一樣的陷阱：**262,144 的 vocab**。logits 是 `seq × bs × V × (2 + 4) bytes`，seq=1024、bs=2 時就是 **3.00 GiB**（= 3.22 GB，注意單位）—— 比 4-bit 權重還大。Week 2 的 H3 就是被這一項稀釋掉梯度檢查點的效益。

**Unsloth 的 fused / chunked cross-entropy 正好是針對這一項。**所以 §4 的框架對照裡，這一項應該是 HF 和 Unsloth 差最多的地方 —— 這是一個事先就能寫下來的預測。

In [ ]:
#@title 3.1 從 config.json 推算，不要用記憶中的規格
import json, math
from huggingface_hub import hf_hub_download

cfg = json.load(open(hf_hub_download(MODEL_IT, "config.json")))
tc = cfg.get("text_config", cfg)

H   = tc["hidden_size"]; L = tc["num_hidden_layers"]
NH  = tc["num_attention_heads"]; NKV = tc.get("num_key_value_heads", NH)
HD  = tc.get("head_dim", H // NH); FF = tc["intermediate_size"]
V   = tc.get("vocab_size", cfg.get("vocab_size"))

print(f"hidden_size {H} | layers {L} | heads {NH} | kv_heads {NKV} | head_dim {HD}")
print(f"intermediate {FF} | vocab {V:,}")

emb   = V * H
attn  = H*NH*HD + 2*(H*NKV*HD) + NH*HD*H
mlp   = 3 * H * FF
per_l = attn + mlp
text  = emb + L*per_l
print(f"\n語言主幹參數：{text/1e9:.3f}B（embed {emb/1e9:.3f}B + {L}×{per_l/1e6:.1f}M）")

# ⚠️ 這個常數不能從 Week 2 直接搬。
# MLX 的 4-bit 是 group_size 64 + bf16 的 scale/bias → 4 + 32/64 = 4.50 bit/param
# （out/gemma4-e4b-tw/model.safetensors 實測 4.501，用它預測誤差是 -0.0%）。
# 這裡用的是 bitsandbytes 的 nf4 + double quant，機制不同（block 64、absmax 再量化一次），
# 名目上也是約 4.5 bit/param，但**要用 §4 量到的峰值回頭校正**，不要當成已知。
BPP_4BIT = 4.5/8
print(f"4-bit 權重     ≈ {text*BPP_4BIT/2**30:.2f} GiB")

def lora_params(rank, attn_only):
    q = H*rank + rank*NH*HD; k = H*rank + rank*NKV*HD
    o = NH*HD*rank + rank*H
    a = q + 2*k + o
    m = 2*(H*rank + rank*FF) + (FF*rank + rank*H)
    return L * (a if attn_only else a + m)

print()
print(f"{'設定':<28}{'可訓練參數':>14}{'Adam 狀態':>14}")
for rank in (8, 16, 32, 64):
    for ao, lbl in ((True,'attn-only'), (False,'all-linear')):
        p = lora_params(rank, ao)
        print(f"  r={rank:<3} {lbl:<18}{p/1e6:>12.2f}M{p*16/2**30:>12.2f} GiB")

print()
print(f"{'seq × bs':<14}{'logits 記憶體':>16}")
for seq in (512, 1024, 2048):
    for bs in (1, 2, 4):
        print(f"  {seq}×{bs:<8}{seq*bs*V*6/2**30:>14.2f} GiB")
print("\n→ 這一項不受梯度檢查點影響（Week 2 H3 實測）。")
print("→ Unsloth 的 fused CE 不會把完整 logits 落地，§4 應該看得到差別。")

json.dump({"hidden": H, "layers": L, "vocab": V, "text_params": text,
           "pred_weights_4bit_gib": text*BPP_4BIT/2**30},
          open(ROOT/'reports'/'memory_prediction_gemma3.json','w'), indent=2)


## 3.2 驗證 Shadow-FT 的前提：base 和 instruct 的權重有多接近？

論文定義相對差距 **σ = Σ|W_B − W_I| / (Σ|W_B| + Σ|W_I|)**，並宣稱所有測過的模型 σ < 0.05。這是整個方法能成立的基礎——如果我們這一對的 σ 很大，Shadow-FT 就不該預期有效。

**這一格是「先驗證前提再做實驗」，不是可有可無的。**用 `safe_open` 逐張量比對，記憶體佔用是常數，不會 OOM。

In [ ]:
#@title 3.2 計算 σ（Shadow-FT Eq.1）
import torch, json
from huggingface_hub import snapshot_download
from safetensors import safe_open
from pathlib import Path

RUN_SIGMA = True  #@param {type:"boolean"}

if RUN_SIGMA:
    def index(repo):
        d = Path(snapshot_download(repo, allow_patterns=["*.safetensors", "*.json"]))
        files = sorted(d.glob("*.safetensors"))
        m = {}
        for f in files:
            with safe_open(f, framework="pt") as h:
                for k in h.keys():
                    m[k] = f
        return m

    print("下載 BASE 與 INSTRUCT 的 fp16 權重（各約 8.6 GB，第一次會久）…")
    ib, ii = index(MODEL_PT), index(MODEL_IT)
    common = sorted(set(ib) & set(ii))
    print(f"BASE {len(ib)} 張量 / INSTRUCT {len(ii)} 張量 / 共同 {len(common)}")
    only_b, only_i = sorted(set(ib)-set(ii)), sorted(set(ii)-set(ib))
    if only_b or only_i: print(f"  ⚠️ 只在 BASE: {only_b[:3]} … 只在 IT: {only_i[:3]} …")

    num = den = 0.0
    per_layer, shape_mismatch = {}, []
    for k in common:
        with safe_open(ib[k], framework="pt") as h: a = h.get_tensor(k).float()
        with safe_open(ii[k], framework="pt") as h: b = h.get_tensor(k).float()
        if a.shape != b.shape:
            shape_mismatch.append(k); continue
        n = (a-b).abs().sum().item(); d = a.abs().sum().item() + b.abs().sum().item()
        num += n; den += d
        per_layer[k] = n/d if d else 0.0
        del a, b

    sigma = num/den
    print(f"\nσ(BASE, INSTRUCT) = {sigma:.4f}")
    print(f"論文 Gemma-3 系列的參考值 < 0.05 → {'✅ 前提成立' if sigma < 0.05 else '⚠️ 偏大，Shadow-FT 的立論要打折'}")
    assert not shape_mismatch, f"形狀不符：{shape_mismatch[:5]} —— 兩個模型不是同一架構，不能做 Shadow-FT"

    top = sorted(per_layer.items(), key=lambda x: -x[1])[:8]
    print(f"\n差最多的 8 個張量：")
    for k, v in top: print(f"  {v:.4f}  {k}")

    json.dump({"sigma": sigma, "n_tensors": len(common),
               "top_divergent": [{"name": k, "sigma": v} for k, v in top]},
              open(ROOT/'reports'/'shadow_ft_sigma.json','w'), indent=2)
    del ib, ii
    import gc; gc.collect()


---
# §4 框架對照：HF transformers+peft vs Unsloth

**這一節是為了回答主管的 Q1。**同一份資料、同一組 LoRA 參數、同樣 30 步，量三件事：峰值記憶體、每步耗時、loss 曲線。

**事先預測**（寫在跑之前，跑完再回來對）：

| 項目 | 預測 | 理由 |
|---|---|---|
| HF + fp16 | **會 NaN** | Gemma 3 layernorm 後 activation 超過 fp16 上限 65,504（廠商說法，本格就是在驗證它） |
| HF + fp32 | 能跑，但**慢 3–5 倍** | T4 fp32 只有 8.1 TFLOPS，fp16 有 65 TFLOPS |
| Unsloth | 能跑，且峰值記憶體**明顯較低** | fused CE 不落地完整 logits（§3.1 算出來是 3.00 GiB） |
| loss | 三者**應該接近** | 若差很多，代表某一邊的數學不對，不是優化 |

最後一列是最重要的對帳：**優化框架的正當性建立在「數值等價」上**。如果 Unsloth 的 loss 曲線和 HF 對不起來，那省下來的記憶體就不能算數。

In [ ]:
#@title 4.1 共用工具：記憶體與計時
import torch, time, json, gc, os
from pathlib import Path

def reset_mem():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def peak_gib():
    return torch.cuda.max_memory_allocated() / 2**30

def free_all(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache()

def done_path(tag): return ROOT / 'results' / f'{tag}.json'
def is_done(tag):   return done_path(tag).exists()
def mark_done(tag, payload):
    done_path(tag).write_text(json.dumps(payload, ensure_ascii=False, indent=2))
    print(f"  → 已寫入 results/{tag}.json")

class StepTimer:
    # 用 TrainerCallback 記錄每步耗時與 loss，取後半平均（前幾步含 warmup 與編譯）。
    def __init__(self): self.t=[]; self.loss=[]; self._last=None
    def make(self):
        from transformers import TrainerCallback
        outer = self
        class CB(TrainerCallback):
            def on_step_begin(self, args, state, control, **kw): outer._last = time.time()
            def on_step_end(self, args, state, control, **kw):
                if outer._last: outer.t.append(time.time() - outer._last)
            def on_log(self, args, state, control, logs=None, **kw):
                if logs and 'loss' in logs: outer.loss.append(logs['loss'])
        return CB()
    def summary(self):
        h = self.t[len(self.t)//2:] or self.t
        return {"s_per_step": sum(h)/len(h) if h else None,
                "n_steps": len(self.t),
                "loss_first": self.loss[0] if self.loss else None,
                "loss_last": self.loss[-1] if self.loss else None,
                "loss_curve": self.loss}
print("ok")

In [ ]:
#@title 4.1b 版本相容層（TRL / Unsloth 的 API 這一年改過名，先探測再用）
import inspect

def make_sft_config(**kw):
    # TRL 新版用 max_length，舊版用 max_seq_length；dataset_text_field 也曾搬家。
    from trl import SFTConfig
    sig = set(inspect.signature(SFTConfig.__init__).parameters)
    if 'max_length' not in sig and 'max_seq_length' in sig and 'max_length' in kw:
        kw['max_seq_length'] = kw.pop('max_length')
    if 'max_seq_length' not in sig and 'max_length' in sig and 'max_seq_length' in kw:
        kw['max_length'] = kw.pop('max_seq_length')
    dropped = {k: kw.pop(k) for k in list(kw) if k not in sig}
    if dropped:
        print(f"  [compat] SFTConfig 不認得，已忽略：{list(dropped)}")
    return SFTConfig(**kw)

def for_inference(model):
    from unsloth import FastModel
    for fn in ('for_inference',):
        f = getattr(FastModel, fn, None)
        if f:
            try:
                return f(model)
            except Exception as e:
                print(f"  [compat] FastModel.{fn} 失敗（{e}），改用 model.eval()")
    model.eval()
    return model

print("相容層就緒")

In [ ]:
#@title 4.2 HF transformers + peft baseline（fp16 → 預期 NaN；再跑 fp32）
import torch, json, math
from datasets import load_dataset

N_STEPS_AB = 30   #@param {type:"integer"}
SEQ_AB     = 1024 #@param {type:"integer"}
BS_AB      = 2    #@param {type:"integer"}

def run_hf(dtype_name):
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer

    reset_mem()
    dtype = {"fp16": torch.float16, "fp32": torch.float32}[dtype_name]
    tk = AutoTokenizer.from_pretrained(MODEL_IT)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    m = AutoModelForCausalLM.from_pretrained(MODEL_IT, quantization_config=bnb,
                                             dtype=dtype, device_map={"": 0},
                                             attn_implementation="eager")
    m.config.use_cache = False
    m.gradient_checkpointing_enable()
    m = get_peft_model(m, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))

    ds = load_dataset("json", data_files=str(ROOT/'data'/'train_inline.jsonl'), split="train")
    timer = StepTimer()
    tr = SFTTrainer(
        model=m, train_dataset=ds, processing_class=tk,
        args=make_sft_config(output_dir=str(SCRATCH/'out'/f'hf_{dtype_name}'),
                       per_device_train_batch_size=BS_AB, gradient_accumulation_steps=1,
                       max_steps=N_STEPS_AB, learning_rate=1e-4, logging_steps=1,
                       max_length=SEQ_AB, optim="adamw_torch",
                       fp16=(dtype_name=="fp16"), bf16=False,
                       gradient_checkpointing=True, report_to="none", seed=42,
                       save_strategy="no", dataset_text_field="text"),
        callbacks=[timer.make()])
    tr.train()
    r = timer.summary(); r.update({"framework": f"HF+peft ({dtype_name})", "peak_gib": peak_gib()})
    r["nan"] = bool(r["loss_last"] is None or (isinstance(r["loss_last"], float)
                    and (math.isnan(r["loss_last"]) or r["loss_last"] == 0.0)))
    free_all(m, tr)
    return r

AB = {}
for dt in ("fp16", "fp32"):
    tag = f"ab_hf_{dt}"
    if is_done(tag):
        AB[dt] = json.loads(done_path(tag).read_text()); print(f"{tag} 已完成，跳過"); continue
    print(f"\n===== HF + {dt} =====")
    try:
        AB[dt] = run_hf(dt)
    except Exception as e:
        AB[dt] = {"framework": f"HF+peft ({dt})", "error": f"{type(e).__name__}: {e}"}
        print(f"  失敗：{AB[dt]['error']}")
    mark_done(tag, AB[dt])
    print(json.dumps({k:v for k,v in AB[dt].items() if k!='loss_curve'}, ensure_ascii=False, indent=2))

In [ ]:
#@title 4.3 Unsloth（同樣參數、同樣 30 步）
#  ⚠️ 這一格跑完請「執行階段 → 重新啟動工作階段」再跑 §5。
#     unsloth 會 patch transformers，和上一格的 HF 混在同一個 process 裡容易出怪事。
import json, math

def run_unsloth():
    from unsloth import FastModel
    from trl import SFTTrainer
    from datasets import load_dataset
    reset_mem()
    model, tk = FastModel.from_pretrained(
        model_name=MODEL_IT, max_seq_length=SEQ_AB, load_in_4bit=True, full_finetuning=False)
    model = FastModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=42,
        finetune_vision_layers=False, finetune_language_layers=True)
    ds = load_dataset("json", data_files=str(ROOT/'data'/'train_inline.jsonl'), split="train")
    timer = StepTimer()
    tr = SFTTrainer(model=model, train_dataset=ds, processing_class=tk,
        args=make_sft_config(output_dir=str(SCRATCH/'out'/'unsloth_ab'),
                       per_device_train_batch_size=BS_AB, gradient_accumulation_steps=1,
                       max_steps=N_STEPS_AB, learning_rate=1e-4, logging_steps=1,
                       max_length=SEQ_AB, optim="adamw_8bit",
                       fp16=False, bf16=False,  # unsloth 自己決定精度
                       report_to="none", seed=42, save_strategy="no",
                       dataset_text_field="text"),
        callbacks=[timer.make()])
    tr.train()
    r = timer.summary(); r.update({"framework": "Unsloth", "peak_gib": peak_gib()})
    r["nan"] = bool(r["loss_last"] is None or (isinstance(r["loss_last"], float) and math.isnan(r["loss_last"])))
    free_all(model, tr)
    return r

tag = "ab_unsloth"
if is_done(tag):
    AB["unsloth"] = json.loads(done_path(tag).read_text()); print("已完成，跳過")
else:
    AB["unsloth"] = run_unsloth(); mark_done(tag, AB["unsloth"])
print(json.dumps({k:v for k,v in AB["unsloth"].items() if k!='loss_curve'}, ensure_ascii=False, indent=2))

In [ ]:
#@title 4.4 框架對照表（貼進報告用）
import json
rows = []
for tag, label in [("ab_hf_fp16","HF+peft fp16"), ("ab_hf_fp32","HF+peft fp32"), ("ab_unsloth","Unsloth")]:
    if is_done(tag):
        rows.append((label, json.loads(done_path(tag).read_text())))

print(f"{'框架':<18}{'峰值記憶體':>12}{'s/step':>10}{'相對速度':>10}{'首 loss':>10}{'末 loss':>10}  備註")
base_s = next((r[1].get('s_per_step') for r in rows if r[0]=='HF+peft fp32' and r[1].get('s_per_step')), None)
for label, r in rows:
    if 'error' in r:
        print(f"{label:<18}{'—':>12}{'—':>10}{'—':>10}{'—':>10}{'—':>10}  {r['error'][:50]}"); continue
    sp = r.get('s_per_step'); rel = f"{base_s/sp:.2f}x" if (sp and base_s) else "—"
    note = "❌ NaN / 未收斂" if r.get('nan') else ""
    print(f"{label:<18}{r['peak_gib']:>11.2f}G{sp:>10.3f}{rel:>10}"
          f"{(r.get('loss_first') or float('nan')):>10.4f}{(r.get('loss_last') or float('nan')):>10.4f}  {note}")

print()
print("要回答主管的三句話：")
print("  1. Week 2 用的是 MLX，不是純 transformers，也不是優化框架。")
print("  2. 這張表就是「有無記憶體優化」的量化差距。")
print("  3. loss 對得起來，代表省的記憶體不是靠犧牲數學換來的。")


---
# §5 LoRA 參數掃描

**這一節回答主管的 Q2。**

## 為什麼這是 Week 3 最重要的一節

翻 `mlx_lm/tuner/lora.py` 之後發現：

```python
return y + (self.scale * z).astype(x.dtype)     # scale 直接乘，沒有除以 rank
```

Week 2 用的 `scale: 20.0` 是**直接乘數**，換算成 PEFT 就是 `lora_alpha = 320`（r=16 → scaling 20）。業界常規是 `alpha=32, r=16` → scaling **2**。

**我們用的強度是常規的 10 倍，而且從來沒有意識到。**

Stage A 就是要驗證：Week 2 那 17.4 個百分點，有多少只是這一件事。

In [ ]:
#@title 5.1 通用訓練函式（Stage A/B/C 與 Shadow-FT 共用）
import json, time, torch, gc
from pathlib import Path

def train_lora(tag, *, model_name, data_file, rank, alpha, steps,
               target="all", seq=1024, bs=2, ga=2, lr=1e-4, seed=42,
               layers=None, save_dir=None):
    # 回傳 dict；adapter 存到 SCRATCH（大檔不進 Drive）。已完成則直接讀回。
    if is_done(f"train_{tag}"):
        print(f"[skip] {tag}")
        return json.loads(done_path(f"train_{tag}").read_text())

    from unsloth import FastModel
    from trl import SFTTrainer
    from datasets import load_dataset

    ATTN = ["q_proj","k_proj","v_proj","o_proj"]
    MLP  = ["gate_proj","up_proj","down_proj"]
    tmods = ATTN if target == "attn" else ATTN + MLP

    save_dir = Path(save_dir or (SCRATCH/'out'/tag))
    reset_mem()
    t0 = time.time()

    model, tk = FastModel.from_pretrained(
        model_name=model_name, max_seq_length=seq, load_in_4bit=True, full_finetuning=False)
    model = FastModel.get_peft_model(
        model, r=rank, lora_alpha=alpha, lora_dropout=0.0, bias="none",
        target_modules=tmods, use_gradient_checkpointing="unsloth",
        random_state=seed, finetune_vision_layers=False, finetune_language_layers=True,
        **({"layers_to_transform": layers} if layers else {}))

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{tag}] 可訓練參數 {n_train/1e6:.2f}M | r={rank} alpha={alpha} "
          f"scaling={alpha/rank:.1f} | target={target} | {steps} 步")

    ds = load_dataset("json", data_files=str(ROOT/'data'/data_file), split="train")
    timer = StepTimer()
    tr = SFTTrainer(model=model, train_dataset=ds, processing_class=tk,
        args=make_sft_config(output_dir=str(save_dir/'_ckpt'),
                       per_device_train_batch_size=bs, gradient_accumulation_steps=ga,
                       max_steps=steps, learning_rate=lr, warmup_ratio=0.05,
                       lr_scheduler_type="cosine", logging_steps=10, max_length=seq,
                       optim="adamw_8bit", fp16=False, bf16=False, report_to="none",
                       seed=seed, save_strategy="no", dataset_text_field="text"),
        callbacks=[timer.make()])
    tr.train()

    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(save_dir)); tk.save_pretrained(str(save_dir))

    rec = timer.summary()
    rec.update({"tag": tag, "model": model_name, "data": data_file, "rank": rank,
                "alpha": alpha, "scaling": alpha/rank, "target": target, "steps": steps,
                "seq": seq, "bs": bs, "grad_accum": ga, "lr": lr,
                "trainable_params": n_train, "peak_gib": peak_gib(),
                "wall_min": (time.time()-t0)/60, "adapter_dir": str(save_dir)})
    mark_done(f"train_{tag}", rec)
    free_all(model, tr)
    return rec
print("ok")


## Stage A — scaling 掃描（最重要的一組）

固定 `r=16`、all-linear、`inline` 資料、200 步，只動 `lora_alpha`。

| run | alpha | scaling | 意義 |
|---|---:|---:|---|
| A1 | 8 | 0.5 | 很保守 |
| A2 | 16 | 1.0 | |
| A3 | 32 | **2.0** | **業界常規** |
| A4 | 64 | 4.0 | |
| A5 | **320** | **20.0** | **重現 Week 2** |

**先寫下預測再跑**：無法解析率隨 scaling 單調上升，A5 應該重現 Week 2 的格式崩潰（~40%），A3 應該 < 5%。

In [ ]:
#@title 5.2 Stage A：scaling 掃描（約 5 × 12 分鐘）
STEPS_SWEEP = 200  #@param {type:"integer"}

STAGE_A = [("A1", 16, 8), ("A2", 16, 16), ("A3", 16, 32), ("A4", 16, 64), ("A5", 16, 320)]
for name, r_, a_ in STAGE_A:
    train_lora(name, model_name=MODEL_IT, data_file='train_inline.jsonl',
               rank=r_, alpha=a_, steps=STEPS_SWEEP, target="all")
print("\nStage A 完成。跑 §7 的評測才會有結論。")

In [ ]:
#@title 5.3 Stage B：rank 掃描（scaling 固定 2.0）
STAGE_B = [("B1", 8, 16), ("B2", 16, 32), ("B3", 32, 64), ("B4", 64, 128)]
for name, r_, a_ in STAGE_B:
    if name == "B2":
        print("[skip] B2 == A3，同一組設定"); continue
    train_lora(name, model_name=MODEL_IT, data_file='train_inline.jsonl',
               rank=r_, alpha=a_, steps=STEPS_SWEEP, target="all")

In [ ]:
#@title 5.4 Stage C：target module（attn-only vs all-linear）
train_lora("C1", model_name=MODEL_IT, data_file='train_inline.jsonl',
           rank=16, alpha=32, steps=STEPS_SWEEP, target="attn")
print("C2 == A3（all-linear），不重跑")


---
# §6 Shadow-FT

**這一節回答主管的 Q4。**

論文：*Shadow-FT: Tuning Instruct Model via Training on Paired Base Model*（arXiv 2505.12716）

```
Step 1:  W_B⁺ = Tune(W_B)                  ← 在 BASE 上訓練
Step 2:  W_I⁺ = W_I + (W_B⁺ − W_B)         ← 把差值搬到 INSTRUCT
```

**LoRA 下 base 項會抵消**：`W_I⁺ = W_I + (W_B + BA − W_B) = W_I + BA`。
所以實作就是「在 `-pt` 上訓練 adapter，然後把同一個 adapter 掛到 `-it` 上」。

## 2×2 設計

| | scaling = 2（常規） | scaling = 20（Week 2） |
|---|---|---|
| **常規 LoRA on `-it`** | A3 | A5 |
| **Shadow-FT（train on `-pt`）** | **D1** | **D2** |

- A5 崩、A3 不崩 → Week 2 的問題主要是超參數
- A3 也崩、D1 不崩 → 主管的判斷成立，Shadow-FT 是解法
- D1 和 D2 差距 << A3 和 A5 的差距 → **假設 S3：Shadow-FT 讓超參數不再那麼要命**（論文沒做這個）

**注意**：訓練 `-pt` 時，資料仍然是用 **`-it` 的 chat template** 渲染的（§2.1 已經做好了）。這一點很重要——delta 必須活在和目標模型同一個座標系裡。

In [ ]:
#@title 6.1 在 BASE 上訓練（D1 / D2）
train_lora("D1", model_name=MODEL_PT, data_file='train_inline.jsonl',
           rank=16, alpha=32,  steps=STEPS_SWEEP, target="all")
train_lora("D2", model_name=MODEL_PT, data_file='train_inline.jsonl',
           rank=16, alpha=320, steps=STEPS_SWEEP, target="all")


## 6.2 移植：把在 BASE 上學到的 delta 加到 INSTRUCT

**不用 `PeftModel.from_pretrained` 自動掛，改成逐張量手動相加。**三個理由：

1. PEFT 的模組命名依賴載入路徑，unsloth 包過一層之後不保證能對上 `-it` 的 state dict。手動做，對不上就直接報錯，不會靜默錯掉。
2. 可以順便實作論文 Eq.6 的縮放係數 α（`W_I⁺ = W_I + α·ΔW`）。
3. 可以順便量每個模組的 `‖ΔW‖ / ‖W‖`，看 delta 到底有多大 —— 這是免費的健全性檢查。

In [ ]:
#@title 6.2 Shadow graft：W_I⁺ = W_I + α·(scaling · B @ A)
import torch, json, re, shutil
from pathlib import Path
from safetensors.torch import load_file, save_file
from safetensors import safe_open
from huggingface_hub import snapshot_download

def shadow_graft(adapter_dir, target_repo, out_dir, alpha_scale=1.0):
    adapter_dir, out_dir = Path(adapter_dir), Path(out_dir)
    if (out_dir/'model.safetensors.index.json').exists() or (out_dir/'model.safetensors').exists():
        print(f"[skip] {out_dir} 已存在"); return out_dir

    cfg = json.loads((adapter_dir/'adapter_config.json').read_text())
    lora_scaling = cfg['lora_alpha'] / cfg['r']
    if cfg.get('use_rslora'): lora_scaling = cfg['lora_alpha'] / (cfg['r'] ** 0.5)
    print(f"adapter: r={cfg['r']} alpha={cfg['lora_alpha']} → LoRA scaling {lora_scaling}")
    print(f"移植縮放 α = {alpha_scale}（論文 Eq.6；1.0 = 標準 Shadow-FT）")

    ad = load_file(adapter_dir/'adapter_model.safetensors')
    pairs = {}
    for k, v in ad.items():
        m = re.match(r'^(?:base_model\.model\.)?(.*)\.lora_(A|B)\.(?:default\.)?weight$', k)
        if not m:
            if 'lora' in k: print(f"  ⚠️ 無法解析的 key：{k}")
            continue
        pairs.setdefault(m.group(1), {})[m.group(2)] = v
    print(f"解析出 {len(pairs)} 個模組的 A/B 對")
    assert pairs, "adapter 裡沒有 lora_A/lora_B —— 檢查 adapter_dir"

    src = Path(snapshot_download(target_repo, allow_patterns=["*.safetensors", "*.json", "*.model", "*.jinja", "*.txt"]))
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        if f.is_file() and f.suffix in ('.json', '.model', '.txt', '.jinja'):
            shutil.copy(f, out_dir/f.name)

    shards = sorted(src.glob('*.safetensors'))
    key2shard = {}
    for f in shards:
        with safe_open(f, framework='pt') as h:
            for k in h.keys(): key2shard[k] = f

    def find_key(mod):
        # PEFT 的模組路徑會少掉 "model." 之類的前綴，逐步放寬比對
        for cand in (f"{mod}.weight", f"model.{mod}.weight"):
            if cand in key2shard: return cand
        suffix = mod.split('base_model.model.')[-1] + '.weight'
        hits = [k for k in key2shard if k.endswith(suffix)]
        return hits[0] if len(hits) == 1 else None

    stats, grafted, missing = [], 0, []
    for f in shards:
        sd = load_file(f)
        for mod, ab in pairs.items():
            k = find_key(mod)
            if k is None: missing.append(mod); continue
            if k not in sd: continue
            W = sd[k]
            A, B = ab['A'].float(), ab['B'].float()          # A:[r,in]  B:[out,r]
            dW = (B @ A) * lora_scaling * alpha_scale
            assert dW.shape == W.shape, f"形狀不符 {mod}: {tuple(dW.shape)} vs {tuple(W.shape)}"
            rel = (dW.abs().sum() / W.float().abs().sum()).item()
            stats.append({"module": mod, "target_key": k, "rel_delta": rel})
            sd[k] = (W.float() + dW).to(W.dtype)
            grafted += 1
        save_file(sd, str(out_dir/f.name), metadata={"format": "pt"})
        del sd

    missing = sorted(set(missing) - {s['module'] for s in stats})
    print(f"\n成功移植 {grafted} / {len(pairs)} 個模組")
    assert grafted == len(pairs), f"有 {len(missing)} 個模組對不上：{missing[:5]} —— 不要當作成功"
    rels = [s['rel_delta'] for s in stats]
    print(f"‖ΔW‖/‖W‖  平均 {sum(rels)/len(rels):.5f}  最大 {max(rels):.5f}  最小 {min(rels):.5f}")
    print(f"（對照：§3.2 量到的 σ(base, instruct) 在同一個數量級才合理）")
    json.dump({"adapter": str(adapter_dir), "target": target_repo, "alpha_scale": alpha_scale,
               "lora_scaling": lora_scaling, "n_grafted": grafted,
               "rel_delta_mean": sum(rels)/len(rels), "per_module": stats},
              open(ROOT/'reports'/f'graft_{out_dir.name}.json','w'), indent=2)
    return out_dir

for tag in ("D1", "D2"):
    rec = json.loads(done_path(f"train_{tag}").read_text())
    shadow_graft(rec['adapter_dir'], MODEL_IT, SCRATCH/'out'/f'{tag}_shadow_merged', alpha_scale=1.0)


## 6.3 對照組：常規 LoRA 也要融合成同樣的形式

**不能拿「融合後的 Shadow-FT」去比「掛 adapter 的常規 LoRA」**——兩者的推論路徑不同（融合過的走一般 linear，掛 adapter 的多一層計算），數值上會有微小差異。要比就兩邊都融合。

In [ ]:
#@title 6.3 把 A3 / A5 也融合成完整權重
def merge_conventional(tag, target_repo=None):
    rec = json.loads(done_path(f"train_{tag}").read_text())
    return shadow_graft(rec['adapter_dir'], target_repo or rec['model'],
                        SCRATCH/'out'/f'{tag}_merged', alpha_scale=1.0)

for tag in ("A3", "A5"):
    merge_conventional(tag)
print("\n現在有四個可比的模型：A3_merged / A5_merged / D1_shadow_merged / D2_shadow_merged")


---
# §7 TMMLU+ 評測

## 為什麼不直接用 twinkle-eval

讀了 `Eval/twinkle_eval/` 的原始碼之後，發現兩件會影響數字可信度的事：

1. **`shuffle_options` 用的是沒設種子的全域 `random`**（整個套件 grep 不到任何 `seed`）。所以 Week 2 的 base 跑和 tuned 跑，**選項順序是不一樣的** —— 那 17.4 pt 裡有一部分是不同題目排列造成的雜訊。
2. **`average_accuracy` 是對「科目」取平均，不是對「題目」取平均。**三科題數 768 / 139 / 129，所以台語（129 題）和台灣地理（768 題）在總分裡權重相同。

下面這支評測器**完全複製 twinkle-eval 的 prompt 組法與 box 解析邏輯**（逐字比對過），但：

- 固定 shuffle 種子 → 每一組實驗看到**完全相同**的題目排列
- 同時報 **macro**（可和 Week 2 對照）和 **micro**（題目加權）
- 同時報 **嚴格**（只認 `\box{X}`）和 **寬鬆**（再接受「答案是 X」）
- 批次生成，比起 OpenAI 端點逐題呼叫快很多

In [ ]:
#@title 7.1 評測器（複製 twinkle-eval 的計分，但把種子固定）
import re, json, random, time, torch
import pandas as pd
from pathlib import Path

# ---- 與 twinkle-eval 的 BoxExtractor 逐字相同 ----
BOX_PATTERNS = [r"\\{1,2}box{([A-Z])}", r"\\{1,2}boxed{([A-Z])}"]
# ---- 與 scripts/analyze_eval.py 的寬鬆解析相同 ----
LENIENT = [r"box\{\s*([ABCD])\s*\}",
           r"(?:答案是|答案為|正確答案是|應該是|選項)\s*[:：]?\s*([ABCD])"]

def extract_strict(s):
    if not s: return None
    for p in BOX_PATTERNS:
        m = re.search(p, s)
        if m: return m.group(1).strip()
    return None

def extract_lenient(s):
    a = extract_strict(s)
    if a: return a
    if not s: return None
    for p in LENIENT:
        m = re.search(p, s)
        if m: return m.group(1).strip().upper()
    return None

def shuffle_options(row, rng):
    # 與 twinkle-eval 同樣「靠選項文字對回正解」，但 rng 由外部傳入 → 可重現。
    opts = [(k, row[k]) for k in "ABCD" if k in row and pd.notna(row[k])]
    if not opts: return None
    gold_text = row.get(str(row['answer']).strip().upper())
    rng.shuffle(opts)
    new = {"question": row['question']}
    for (old, text), newk in zip(opts, "ABCD"):
        new[newk] = text
        if text == gold_text: new['answer'] = newk
    return new if 'answer' in new else None

def build_prompt(q):
    # twinkle-eval evaluator.py:918 的組法
    return q['question'] + "\n" + "\n".join(f"{k}: {v}" for k, v in q.items()
                                            if k not in ("question", "answer"))

@torch.no_grad()
def evaluate(model_path, tag, subjects=None, limit_per_subject=None,
             max_new_tokens=512, batch_size=8, seed=42, load_4bit=True):
    if is_done(f"eval_{tag}"):
        print(f"[skip] eval_{tag}")
        return json.loads(done_path(f"eval_{tag}").read_text())

    from unsloth import FastModel
    subjects = subjects or EVAL_SUBJECTS
    reset_mem(); t0 = time.time()
    model, tk = FastModel.from_pretrained(model_name=str(model_path), max_seq_length=2048,
                                          load_in_4bit=load_4bit, full_finetuning=False)
    for_inference(model)
    tk.padding_side = "left"
    if tk.pad_token is None: tk.pad_token = tk.eos_token

    per_subject, records = {}, []
    for s in subjects:
        df = pd.read_parquet(TMMLU_DIR / f"{s}.parquet")
        rng = random.Random(seed)          # ← 每一科都從同一個種子開始
        raw = [shuffle_options(r, rng) for _, r in df.iterrows()]
        qs = [q for q in raw if q]
        n_dropped = len(raw) - len(qs)
        if n_dropped:
            # twinkle-eval 會留著這些題；我們丟掉，所以要記下來，否則 n_questions 對不上 1,036
            print(f"    ⚠️ {s}: {n_dropped} 題因選項文字重複/缺失而無法對回正解，已剔除")
        if limit_per_subject: qs = qs[:limit_per_subject]

        n_ok_s = n_ok_l = n_unparsed = 0
        for i in range(0, len(qs), batch_size):
            batch = qs[i:i+batch_size]
            texts = []
            for q in batch:
                user = build_prompt(q)
                msgs = ([{"role":"system","content":SYS_BOX}] if HAS_SYSTEM else []) + \
                       [{"role":"user","content": (user if HAS_SYSTEM else SYS_BOX+"\n\n"+user)}]
                texts.append(tk.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
            enc = tk(texts, return_tensors="pt", padding=True, truncation=True,
                     max_length=1536, add_special_tokens=False).to("cuda")
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 pad_token_id=tk.pad_token_id)
            gen = out[:, enc['input_ids'].shape[1]:]
            for q, g in zip(batch, gen):
                txt = tk.decode(g, skip_special_tokens=True)
                ps, pl = extract_strict(txt), extract_lenient(txt)
                ok_s, ok_l = (ps == q['answer']), (pl == q['answer'])
                n_ok_s += ok_s; n_ok_l += ok_l; n_unparsed += (ps is None)
                records.append({"subject": s, "question": q['question'][:200],
                                "gold": q['answer'], "pred_strict": ps, "pred_lenient": pl,
                                "correct_strict": bool(ok_s), "correct_lenient": bool(ok_l),
                                "n_gen_tokens": int((g != tk.pad_token_id).sum()),
                                "output": txt[:1500]})
            if i % (batch_size*10) == 0:
                print(f"    {s} {i+len(batch)}/{len(qs)}  嚴格 {n_ok_s/(i+len(batch)):.3f}  "
                      f"無法解析 {n_unparsed/(i+len(batch)):.3f}", flush=True)

        n = len(qs)
        per_subject[s] = {"n": n, "n_dropped": n_dropped,
                          "acc_strict": n_ok_s/n, "acc_lenient": n_ok_l/n,
                          "unparsed_rate": n_unparsed/n}
        print(f"  {s:<30} n={n:<5} 嚴格 {n_ok_s/n:.4f}  寬鬆 {n_ok_l/n:.4f}  "
              f"無法解析 {n_unparsed/n:.4f}")

    tot = sum(v['n'] for v in per_subject.values())
    res = {
        "tag": tag, "model": str(model_path), "n_questions": tot, "seed": seed,
        "macro_acc_strict":   sum(v['acc_strict']    for v in per_subject.values())/len(per_subject),
        "macro_acc_lenient":  sum(v['acc_lenient']   for v in per_subject.values())/len(per_subject),
        "macro_unparsed":     sum(v['unparsed_rate'] for v in per_subject.values())/len(per_subject),
        "micro_acc_strict":   sum(v['acc_strict']*v['n']    for v in per_subject.values())/tot,
        "micro_acc_lenient":  sum(v['acc_lenient']*v['n']   for v in per_subject.values())/tot,
        "micro_unparsed":     sum(v['unparsed_rate']*v['n'] for v in per_subject.values())/tot,
        "per_subject": per_subject, "minutes": (time.time()-t0)/60,
    }
    with (ROOT/'results'/f'eval_{tag}.jsonl').open('w') as f:
        for r in records: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    mark_done(f"eval_{tag}", res)
    free_all(model)
    return res
print("ok")

In [ ]:
#@title 7.2 防呆：正式評測前先送 3 題，確認抽得出答案
#  Week 2 有一次評測請求全部成功、解析率卻是 0%，浪費了一整晚。
from unsloth import FastModel
import pandas as pd, random

def probe(model_path, n=3):
    model, tk = FastModel.from_pretrained(model_name=str(model_path), max_seq_length=2048,
                                          load_in_4bit=True, full_finetuning=False)
    for_inference(model); tk.padding_side="left"
    if tk.pad_token is None: tk.pad_token = tk.eos_token
    df = pd.read_parquet(TMMLU_DIR / f"{EVAL_SUBJECTS[0]}.parquet")
    rng = random.Random(42)
    qs = [q for q in (shuffle_options(r, rng) for _, r in df.head(n).iterrows()) if q]
    ok = 0
    for q in qs:
        user = build_prompt(q)
        msgs = ([{"role":"system","content":SYS_BOX}] if HAS_SYSTEM else []) + \
               [{"role":"user","content": (user if HAS_SYSTEM else SYS_BOX+"\n\n"+user)}]
        t = tk.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = tk(t, return_tensors="pt", add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=512, do_sample=False, pad_token_id=tk.pad_token_id)
        txt = tk.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        got = extract_strict(txt)
        print(f"  正解 {q['answer']} | 抽到 {got} | 輸出前 120 字: {txt[:120]!r}")
        ok += got is not None
    free_all(model)
    assert ok > 0, "3 題一題都抽不出 \\box{} —— 先修 prompt，不要開跑正式評測"
    print(f"防呆通過（{ok}/{len(qs)} 題抽得出答案）")

probe(MODEL_IT)

In [ ]:
#@title 7.3 快速評測（每科 100 題）—— 用來掃參數
QUICK_N = 100  #@param {type:"integer"}

QUICK_TARGETS = [("base_it", MODEL_IT)]
for tag in ("A1","A2","A3","A4","A5","B1","B3","B4","C1"):
    if is_done(f"train_{tag}"):
        QUICK_TARGETS.append((tag, json.loads(done_path(f"train_{tag}").read_text())['adapter_dir']))

for tag, path in QUICK_TARGETS:
    print(f"\n===== quick eval: {tag} =====")
    evaluate(path, f"quick_{tag}", limit_per_subject=QUICK_N, batch_size=8)

In [ ]:
#@title 7.4 完整評測（1,036 題，只跑決選的四個 + baseline）
FULL_TARGETS = [
    ("base_it",   MODEL_IT),
    ("A3_conv",   SCRATCH/'out'/'A3_merged'),
    ("A5_conv",   SCRATCH/'out'/'A5_merged'),
    ("D1_shadow", SCRATCH/'out'/'D1_shadow_merged'),
    ("D2_shadow", SCRATCH/'out'/'D2_shadow_merged'),
]
for tag, path in FULL_TARGETS:
    if not Path(str(path)).exists() and not str(path).startswith(("unsloth/","google/")):
        print(f"[skip] {tag}：{path} 不存在"); continue
    print(f"\n===== full eval: {tag} =====")
    evaluate(path, f"full_{tag}", batch_size=8)


---
# §8 彙整

In [ ]:
#@title 8.1 Stage A：scaling vs 格式保留（Week 3 的主圖）
import json, matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt

rows = []
for name, r_, a_ in STAGE_A:
    p = done_path(f"quick_{name}")
    if not p.exists(): continue
    e = json.loads(p.read_text()); t = json.loads(done_path(f"train_{name}").read_text())
    rows.append({"run": name, "scaling": t['scaling'], "alpha": a_,
                 "unparsed": e['micro_unparsed'], "strict": e['micro_acc_strict'],
                 "lenient": e['micro_acc_lenient'], "loss": t['loss_last']})

b = json.loads(done_path("quick_base_it").read_text()) if is_done("quick_base_it") else None

print(f"{'run':<5}{'alpha':>7}{'scaling':>9}{'無法解析':>10}{'嚴格':>9}{'寬鬆':>9}{'train loss':>12}")
if b: print(f"{'base':<5}{'—':>7}{'—':>9}{b['micro_unparsed']:>10.3f}"
            f"{b['micro_acc_strict']:>9.3f}{b['micro_acc_lenient']:>9.3f}{'—':>12}")
for r in rows:
    print(f"{r['run']:<5}{r['alpha']:>7}{r['scaling']:>9.1f}{r['unparsed']:>10.3f}"
          f"{r['strict']:>9.3f}{r['lenient']:>9.3f}{(r['loss'] or 0):>12.4f}")

if rows:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    x = [r['scaling'] for r in rows]
    ax[0].plot(x, [r['unparsed'] for r in rows], 'o-', color='crimson')
    if b: ax[0].axhline(b['micro_unparsed'], ls='--', c='gray', label='base (no FT)')
    ax[0].set_xscale('log'); ax[0].set_xlabel('LoRA scaling (alpha / r)')
    ax[0].set_ylabel('unparsed rate'); ax[0].set_title('format collapse vs LoRA strength')
    ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(x, [r['strict'] for r in rows], 'o-', label='strict')
    ax[1].plot(x, [r['lenient'] for r in rows], 's-', label='lenient')
    if b:
        ax[1].axhline(b['micro_acc_strict'], ls='--', c='gray', label='base strict')
    ax[1].set_xscale('log'); ax[1].set_xlabel('LoRA scaling (alpha / r)')
    ax[1].set_ylabel('accuracy'); ax[1].set_title('accuracy vs LoRA strength')
    ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(ROOT/'reports'/'stageA_scaling.png', dpi=140)
    plt.show()
    print("\n判讀：")
    print("  無法解析率隨 scaling 單調上升 → Week 2 的崩潰主因是超參數。")
    print("  若 scaling=2 也崩            → 才是模型 post-training 的問題（Q3）。")

In [ ]:
#@title 8.2 2×2：常規 LoRA vs Shadow-FT
import json
grid = [("A3_conv","常規 LoRA","2.0"), ("A5_conv","常規 LoRA","20.0"),
        ("D1_shadow","Shadow-FT","2.0"), ("D2_shadow","Shadow-FT","20.0")]
print(f"{'方法':<12}{'scaling':>8}{'嚴格(micro)':>13}{'寬鬆':>9}{'無法解析':>10}{'嚴格(macro)':>13}")
if is_done("full_base_it"):
    b = json.loads(done_path("full_base_it").read_text())
    print(f"{'未微調 base':<12}{'—':>8}{b['micro_acc_strict']:>13.4f}"
          f"{b['micro_acc_lenient']:>9.4f}{b['micro_unparsed']:>10.4f}{b['macro_acc_strict']:>13.4f}")
for tag, meth, sc in grid:
    if not is_done(f"full_{tag}"): print(f"{meth:<12}{sc:>8}   （未跑）"); continue
    e = json.loads(done_path(f"full_{tag}").read_text())
    print(f"{meth:<12}{sc:>8}{e['micro_acc_strict']:>13.4f}{e['micro_acc_lenient']:>9.4f}"
          f"{e['micro_unparsed']:>10.4f}{e['macro_acc_strict']:>13.4f}")

print()
print("三條假設的判讀：")
print("  S1  Shadow-FT 的無法解析率 < 5%，且明顯低於同 scaling 的常規 LoRA")
print("  S2  Shadow-FT 的嚴格正確率 >= 未微調 base")
print("  S3  |D1 - D2| << |A3 - A5|  -> Shadow-FT 對超參數不敏感（論文沒做這一條）")

In [ ]:
#@title 8.3 匯出全部結果為一份 Markdown
import json, datetime
from pathlib import Path

lines = ["# Week 3 實驗結果（自動產生）", "",
         f"產生時間：{datetime.datetime.now():%Y-%m-%d %H:%M}", "",
         "> 所有數字由 `notebooks/week3_colab.ipynb` 產生，原始 JSON 在 `results/`。", ""]

def table(title, tags, cols, hdr):
    lines.append(f"## {title}"); lines.append("")
    lines.append("| " + " | ".join(hdr) + " |")
    lines.append("|" + "|".join(["---"]*len(hdr)) + "|")
    for t in tags:
        p = done_path(t)
        if not p.exists(): continue
        d = json.loads(p.read_text())
        lines.append("| " + " | ".join(
            f"{d.get(c):.4f}" if isinstance(d.get(c), float) else str(d.get(c, "—"))
            for c in cols) + " |")
    lines.append("")

table("框架對照（§4）", ["ab_hf_fp16","ab_hf_fp32","ab_unsloth"],
      ["framework","peak_gib","s_per_step","loss_last","nan"],
      ["框架","峰值 GiB","s/step","末 loss","NaN"])
table("訓練設定（§5–6）",
      [f"train_{t}" for t in ("A1","A2","A3","A4","A5","B1","B3","B4","C1","D1","D2")],
      ["tag","model","rank","alpha","scaling","target","trainable_params","peak_gib","wall_min","loss_last"],
      ["run","模型","r","alpha","scaling","target","可訓練參數","峰值 GiB","分鐘","末 loss"])
table("快速評測（每科 100 題）",
      [f"quick_{t}" for t in ("base_it","A1","A2","A3","A4","A5","B1","B3","B4","C1")],
      ["tag","micro_acc_strict","micro_acc_lenient","micro_unparsed","macro_acc_strict"],
      ["run","嚴格(micro)","寬鬆(micro)","無法解析","嚴格(macro)"])
table("完整評測（1,036 題）",
      [f"full_{t}" for t in ("base_it","A3_conv","A5_conv","D1_shadow","D2_shadow")],
      ["tag","n_questions","micro_acc_strict","micro_acc_lenient","micro_unparsed","macro_acc_strict","minutes"],
      ["run","題數","嚴格(micro)","寬鬆(micro)","無法解析","嚴格(macro)","分鐘"])

out = ROOT/'reports'/'week3_results.md'
out.write_text("\n".join(lines))
print(out); print(); print("\n".join(lines))


---
# §9 收尾檢查

跑完之後，**這幾個數字要能對得起來**，對不上就是哪裡錯了：

| 對帳項 | 條件 |
|---|---|
| `σ(base, instruct)`（§3.2） | < 0.05，否則 Shadow-FT 的前提不成立 |
| graft 的 `n_grafted` | **必須等於** adapter 裡的模組數，少一個都不行（程式已 assert） |
| `‖ΔW‖/‖W‖`（§6.2） | 應該和 σ 在同一個數量級。大很多代表 LoRA 太強了 |
| 未微調 base 的無法解析率 | < 5%。若一開始就高，是 prompt 有問題不是模型 |
| HF fp32 與 Unsloth 的 loss 曲線 | 應該接近。差很多代表某一邊數學不對 |
| 快速評測（100 題）與完整評測（1,036 題）的排序 | 應該一致。不一致代表 100 題的雜訊太大，掃描結論不可信 |

## 下載結果回本機

```bash
# 在本機 repo 執行
rsync -av ~/Google\ Drive/My\ Drive/ultrascale-lab-week3/results/  results/week3/
rsync -av ~/Google\ Drive/My\ Drive/ultrascale-lab-week3/reports/  reports/week3/
```

或直接用下一格打包下載。

In [ ]:
#@title 9.1 打包結果（不含大檔）
import shutil, os
pkg = '/content/week3_results'
shutil.rmtree(pkg, ignore_errors=True)
os.makedirs(pkg)
for d in ('results', 'reports'):
    shutil.copytree(ROOT/d, f'{pkg}/{d}', dirs_exist_ok=True)
shutil.make_archive('/content/week3_results', 'zip', pkg)
print("大小:", os.path.getsize('/content/week3_results.zip')/1e6, "MB")
try:
    from google.colab import files
    files.download('/content/week3_results.zip')
except Exception as e:
    print("自動下載失敗，從左側檔案面板手動下載 /content/week3_results.zip")